<a href="https://colab.research.google.com/github/p-tech/ntu-growth-hacking/blob/main/sentiment/Notebook_3_Oddbox_Stars_and_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 · Do the stars tell the same story as the words?

**NTU MSc Growth Hacking | Customer Insight Lab | Oddbox**

The founder wants to know what customers value and what might stop them ordering
again. A colleague suggests reading five-star reviews for strengths and one-star
reviews for problems. What might that approach miss?

In this workshop you will analyse review text using TextBlob, VADER and RoBERTa.
You will then bring in the customer's own star rating, change which reviews are
included, and export two groups for topic modelling in Notebook 5.

**Work down the notebook one cell at a time.** Save your own copy before starting.
You need the class review file in CSV or Excel format and an internet connection
for the first model download.

## 1 · What is in the review file?

Upload the Oddbox review file. One row should represent one customer review.
In the menus, choose the column containing the **customer's own words** and the
column containing their **one-to-five-star rating**. A business reply can express
a very different sentiment from the review it answers.

If the file has a unique review ID, select it. A customer ID is unsuitable because
one customer can write several reviews.

In [ ]:
%pip -q install textblob vaderSentiment "transformers>=5,<6" "huggingface-hub>=1.32,<3" openpyxl

In [ ]:
import io
import re
import zipfile
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one CSV or Excel file.")

filename, payload = next(iter(uploaded.items()))
if filename.lower().endswith(".xlsx"):
    raw = pd.read_excel(io.BytesIO(payload), dtype=str, keep_default_na=False)
elif filename.lower().endswith(".csv"):
    raw = pd.read_csv(io.BytesIO(payload), dtype=str, keep_default_na=False,
                      encoding="utf-8-sig", sep=None, engine="python")
else:
    raise ValueError("Choose a CSV or .xlsx file.")

print(f"{len(raw)} imported rows. Preview the headings before choosing.")
display(raw.head(3))
columns = list(raw.columns)
if len(columns) < 2:
    raise ValueError("The file needs separate columns for review text and stars.")

def column_menu(description, suggestions, optional=False):
    options = ([("No review ID", None)] if optional else []) + [(c, c) for c in columns]
    normalised = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in columns}
    guess = next((normalised[s] for s in suggestions if s in normalised), options[0][1])
    menu = widgets.Dropdown(options=options, value=guess, description=description)
    display(menu)
    return menu

text_menu = column_menu("Review text", ["reviewtext", "text", "review", "content"])
star_menu = column_menu("Stars", ["rating", "stars", "starrating", "reviewrating"])
id_menu = column_menu("Review ID", ["reviewid", "id"], optional=True)

## 2 · Which reviews can we use fairly?

A blank review can still have five stars. It tells us how the customer rated the
experience, but gives the language models nothing to read. A review with words
and no valid rating presents the opposite problem. For the comparison below,
we need both pieces of evidence.

The cell keeps the original wording. It trims spare spaces and removes repeated
imports where the review ID, wording and rating all match. Without a reliable ID,
identical wording is flagged for inspection and retained. Two customers may
genuinely write “Great service”.

We keep punctuation, emojis and words such as *not*. In Notebook 2 you explored
how changing those clues can change a label. Here we want to change **which
ratings we include** while keeping the wording constant.

In [ ]:
chosen_columns = [v for v in (text_menu.value, star_menu.value, id_menu.value) if v is not None]
if len(set(chosen_columns)) != len(chosen_columns):
    raise ValueError("Choose a different column for each field.")

data = pd.DataFrame({
    "source_row": range(1, len(raw) + 1),
    "original_text": raw[text_menu.value],
    "rating_as_uploaded": raw[star_menu.value],
})
data["review_text"] = data.original_text.str.split().str.join(" ")
data["review_id"] = raw[id_menu.value].str.strip() if id_menu.value else ""

# Accept 4, 4.0, '4 stars', '4/5' and '4 out of 5'. Reject out-of-range values.
pattern = r"^\s*([1-5])(?:\.0+)?(?:\s*(?:stars?|/\s*5|out of\s+5))?\s*$"
data["stars"] = pd.to_numeric(
    data.rating_as_uploaded.str.lower().str.extract(pattern, expand=False),
    errors="coerce"
)
blank = data.review_text.eq("")
bad_rating = data.stars.isna()
eligible = data.loc[~blank & ~bad_rating].copy()
repeat = eligible.review_id.ne("") & eligible.duplicated(["review_id", "review_text", "stars"])
reviews = eligible.loc[~repeat].copy().reset_index(drop=True)
reviews["stars"] = reviews.stars.astype(int)

print(f"Imported rows: {len(data)}")
print(f"Blank text: {blank.sum()} | Missing or invalid rating: {bad_rating.sum()}")
print(f"Rows missing either requirement: {(blank | bad_rating).sum()}")
print(f"Repeated eligible imports removed: {repeat.sum()} | Usable reviews: {len(reviews)}")
if bad_rating.any():
    print("Examples of rating values to check:", data.loc[bad_rating, "rating_as_uploaded"].unique()[:5])
if reviews.empty:
    raise ValueError("No usable reviews. Check the selected headings and rating values.")
repeated_words = reviews.review_text.duplicated(keep=False)
print(f"Retained rows sharing wording with another review: {repeated_words.sum()}")
display(reviews.loc[repeated_words, ["source_row", "review_id", "review_text"]].head(4))

# Withhold stars from the first reading.
display(reviews[["source_row", "review_text"]].head(3))

**Before revealing the stars**, give each of the three displayed reviews one
overall label in your notes: positive, neutral or negative. Choose one review
and predict its star rating. Which words led you there? How easily could someone
give the same wording a different number of stars?

## 3 · What do the three methods hear in the wording?

All three methods receive the review text alone. Each assigns an overall label.
Use the extra scores to understand how it reached that label.

| Method | How to read the result |
|---|---|
| **TextBlob** | Polarity runs from −1 to +1. This exercise labels values below zero negative, above zero positive and exactly zero neutral. Subjectivity runs from 0 to 1. A value of 0.80 suggests opinion-based wording. It does not mean the label is 80% certain. |
| **VADER** | Positive, neutral and negative shares show the balance of signals in the wording. The compound score brings those signals together with rules about emphasis and negation. Scores of +0.05 and above are positive. Scores of −0.05 and below are negative. The middle band is neutral. |
| **RoBERTa** | The model gives three scores that sum to about 1 and selects the highest. If negative scores 0.46 and neutral 0.44, the winning margin is just 0.02. Read the sentence before drawing a firm conclusion. |

The first model download can take several minutes. RoBERTa processes the reviews
in small batches. For long reviews it reads the first 512 tokens, including
special tokens. A verdict at the end of a long review may therefore be missed.

In [ ]:
import torch
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

vader = SentimentIntensityAnalyzer()
methods = ["TextBlob", "VADER", "RoBERTa"]
labels = ["negative", "neutral", "positive"]

def word_scores(text):
    blob = TextBlob(text).sentiment
    v = vader.polarity_scores(text)
    return {
        "TextBlob": "positive" if blob.polarity > 0 else "negative" if blob.polarity < 0 else "neutral",
        "blob_polarity": blob.polarity, "blob_subjectivity": blob.subjectivity,
        "VADER": "positive" if v["compound"] >= .05 else "negative" if v["compound"] <= -.05 else "neutral",
        **{"vader_" + k: value for k, value in v.items()},
    }

results = reviews.copy()
word_table = pd.DataFrame(results.review_text.map(word_scores).tolist())
results[word_table.columns] = word_table

name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()
model_labels = [model.config.id2label[i].lower() for i in range(3)]
if set(model_labels) != set(labels):
    raise ValueError("Unexpected model labels. Check the model name.")

texts = [re.sub(r"https?://\S+", "http", re.sub(r"(?<!\w)@\w+", "@user", t))
         for t in results.review_text]
scores = []
with torch.inference_mode():
    for start in range(0, len(texts), 8):
        inputs = tokenizer(texts[start:start+8], padding=True, truncation=True,
                           max_length=512, return_tensors="pt").to(device)
        scores.extend(model(**inputs).logits.softmax(dim=1).cpu().tolist())

probs = pd.DataFrame(scores, columns=model_labels)
results["RoBERTa"] = probs.idxmax(axis=1)
results["roberta_margin"] = probs.apply(lambda r: r.nlargest(2).diff().abs().iloc[-1], axis=1)
results[["roberta_" + c for c in probs.columns]] = probs
results["models_differ"] = results[methods].nunique(axis=1).gt(1)

display(results[["source_row", "review_text"] + methods].head(3))

## 4 · Reveal the stars. Where do the words tell a different story?

For this exercise we group **1–2 stars as negative, 3 stars as neutral and
4–5 stars as positive**. This is our comparison rule. Customers can use three
stars to express mixed feelings, disappointment or a perfectly adequate experience.

Imagine a five-star review saying, “The delivery was late, but customer service
sorted it in minutes.” The stars describe the customer's overall judgement.
The words mention both a failure and a recovery. A disagreement is worth reading
before deciding that a person or a model got it wrong.

First check your prediction. Then choose a review where the stars and the words
appear to pull in different directions.

In [ ]:
results["star_band"] = results.stars.map({
    1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"
})
for method in methods:
    results[method + "_differs_from_stars"] = results[method].ne(results.star_band)
results["any_star_mismatch"] = results[[m + "_differs_from_stars" for m in methods]].any(axis=1)

pd.set_option('display.max_colwidth', None)

detail = ["source_row", "stars", "star_band", "review_text"] + methods
print("The first three reviews, now with their stars")
display(results[detail].head(3))
print("How many usable reviews have each rating?")
display(results.stars.value_counts().reindex(range(1, 6), fill_value=0).sort_index().rename("reviews").to_frame())
print("Examples where at least one model differs from the star band")
mismatches = results.loc[results.any_star_mismatch, detail]
display(mismatches.head(8))
if mismatches.empty:
    print("All model labels match the star bands in this dataset.")

**Discuss with a neighbour.** Which phrase might explain one mismatch? Does the
review discuss more than one part of the experience? Could a model have missed
sarcasm, a qualification or the customer's final verdict? Ground your answer in
the words you can see.

## 5 · How does selecting the stars change your headline?

Start with all five ratings selected. Then try **5 stars only**, followed by
**1 and 2 stars**. Keep the sentiment method the same while comparing these runs.

Each time you change the star filter, ask:

**How many reviews am I looking at, and how many did the model label positive?**

Imagine you select only five-star reviews and find five. The model labels four of them positive, so your result is 4 out of 5 (80%).

If it labelled just one fewer review positive, the result would fall to 3 out of 5 (60%). With so few reviews, one review makes a big difference.

When you report your finding, say **“The model labelled four of the five five-star reviews positive.”**

That tells the reader whose reviews you selected and what the model found in their words. Then include the lower star ratings and ask: Does the picture change when we hear from more reviewers?

In [ ]:
# Tick the ratings to include. Run this cell again after changing your choices.
one_star = True #@param {type:"boolean"}
two_stars = True #@param {type:"boolean"}
three_stars = True #@param {type:"boolean"}
four_stars = True #@param {type:"boolean"}
five_stars = True #@param {type:"boolean"}

chosen = [i for i, keep in enumerate(
    [one_star, two_stars, three_stars, four_stars, five_stars], start=1
) if keep]
selected = results.loc[results.stars.isin(chosen)].copy()

def summarise(frame, group):
    rows = []
    for method in methods:
        counts = frame[method].value_counts()
        rows.append({
            "group": group, "method": method, "reviews": len(frame),
            **{label: int(counts.get(label, 0)) for label in labels},
            "positive %": round(100 * counts.get("positive", 0) / len(frame), 1) if len(frame) else None,
        })
    return rows

print("Selected ratings:", chosen)
display(pd.DataFrame(summarise(results, "All usable reviews") +
                     summarise(selected, "Selected ratings")))
if selected.empty:
    print("No reviews match. Change the ratings and rerun before exporting.")
else:
    print("Selected reviews where the models differ or a model differs from the stars")
    display(selected.loc[selected.models_differ | selected.any_star_mismatch, detail].head(8))

## 6 · Which reviews should go into the positive and negative files?

Choose the rule that fits your research question. The export uses **the ratings
currently selected in Block 5**. Select all five again if you want both files
to draw from the full dataset.

| Export rule | What question does it help answer? |
|---|---|
| **Customer stars** | What do high-rating and low-rating customers mention, whatever tone they use? |
| **One sentiment model** | What do reviews with positive or negative wording mention according to that method? |
| **Stars and chosen model agree** | Which reviews contain aligned signals? This makes the groups narrower and can leave out revealing mixed experiences. |

For example, a two-star review that RoBERTa labels positive enters the negative
file under **Customer stars**, the positive file under **One sentiment model**
with RoBERTa selected, and neither file under **Stars and chosen model agree**.
Predict what your choice will leave out before running the cell.

In [ ]:
rule = "Customer stars" #@param ["Customer stars", "One sentiment model", "Stars and chosen model agree"]
method = "RoBERTa" #@param ["TextBlob", "VADER", "RoBERTa"]

if selected.empty:
    raise ValueError("Select ratings with at least one review in Block 5 and rerun that cell.")
out = selected.copy()
if rule == "Customer stars":
    assignment = out.star_band
elif rule == "One sentiment model":
    assignment = out[method]
else:
    assignment = out.star_band.where(out.star_band.eq(out[method]), "signals differ")

out["export_group"] = assignment
out["export_rule"] = rule
out["export_method"] = "customer rating only" if rule == "Customer stars" else method
out["selected_ratings"] = ", ".join(map(str, chosen))

groups = {label: out.loc[out.export_group.eq(label)] for label in ["positive", "negative"]}
for label, group in groups.items():
    print(f"{label.title()} file: {len(group)} reviews")
print(f"Outside both files: {len(out) - sum(len(g) for g in groups.values())}")
if any(g.empty for g in groups.values()):
    print("One file has no reviews. Check your rating filter and export rule before topic modelling.")

archive = "Oddbox-Notebook-3-review-exports.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for label, group in groups.items():
        bundle.writestr(f"Oddbox-{label}-reviews.csv",
                        group.to_csv(index=False).encode("utf-8-sig"))
    bundle.writestr("Oddbox-selected-review-audit.csv",
                    out.to_csv(index=False).encode("utf-8-sig"))
files.download(archive)

## 7 · What would you tell the founder?

Complete these three sentences using your results.

1. “Across all ___ usable reviews, ___ labelled ___% positive. When we kept only
   ___ stars, the positive share was ___%, based on ___ reviews.”
2. “Review ___ received ___ stars, but the phrase ‘___’ helps explain why ___
   labelled the wording ___.”
3. “For topic modelling we chose ___ as the export rule because we want to find
   out ___. One type of customer experience this could leave out is ___.”

**For Notebook 5**, unzip the download and keep the positive and negative CSVs.
Each has a **review_text** column, the original rating, the three model labels and
the rule that placed the review in that group. The audit CSV also retains selected
reviews that fell outside both groups.